In [1]:
# ==============================================================================
# CONFIGURATION INITIALE ET IMPORTATIONS
# ==============================================================================
import os
import math
import re
import json
from glob import glob
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications import MobileNetV2

# Reproductibilité
np.random.seed(42)
tf.random.set_seed(42)

# Chemins d'accès au dataset
DATA_ROOT = Path("data/cats_dogs")
train_dir = (DATA_ROOT / "train" / "train") if (DATA_ROOT / "train" / "train").exists() else (DATA_ROOT / "train")
test_dir  = (DATA_ROOT / "test"  / "test")  if (DATA_ROOT / "test"  / "test").exists()  else (DATA_ROOT / "test")

# Paramètres généraux (Ajustés pour MobileNetV2)
IMG_HEIGHT, IMG_WIDTH = 180, 180
batch_size = 32
seed = 1337

# ==============================================================================
# 1. PIPELINE DE CHARGEMENT ET DATA AUGMENTATION
# ==============================================================================
def build_df_from_folder(folder: Path, labeled: bool=True):
    exts = ('*.jpg', '*.jpeg', '*.png', '*.bmp')
    files = []
    for ex in exts:
        files.extend(glob(str(folder / '**' / ex), recursive=True))
    if not files:
        raise FileNotFoundError(f"Aucune image trouvée sous {folder}")
    rows = []
    for f in files:
        if labeled:
            name = Path(f).name.lower()
            parent = Path(f).parent.name.lower()
            if parent in {"cat", "cats"}:
                label = "cat"
            elif parent in {"dog", "dogs"}:
                label = "dog"
            else:
                if re.search(r'(^|[^a-z])cat([^a-z]|$)', name): label = "cat"
                elif re.search(r'(^|[^a-z])dog([^a-z]|$)', name): label = "dog"
                else: continue
            rows.append({"filepath": f, "label": label})
        else:
            rows.append({"filepath": f})
    return pd.DataFrame(rows)

# Construction des structures de données
df_train_full = build_df_from_folder(train_dir, labeled=True)
df_test_full  = build_df_from_folder(test_dir,  labeled=False)

# Split Stratifié 80/20 (Entraînement / Validation)
df_tr, df_val = train_test_split(
    df_train_full, test_size=0.2, stratify=df_train_full["label"], random_state=seed
)

# Application de la Data Augmentation (Uniquement sur le Train)
# Note : MobileNetV2 attend des pixels entre [-1, 1], l'augmentation intègre ce preprocessing
train_gen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
    rotation_range=30,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.3,
    horizontal_flip=True,
)

# Validation et Test : Pas d'augmentation, juste le preprocessing MobileNet
val_gen = ImageDataGenerator(preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input)
test_gen = ImageDataGenerator(preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input)

# Générateurs de flux
train_flow = train_gen.flow_from_dataframe(
    df_tr, x_col="filepath", y_col="label",
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode="binary", batch_size=batch_size,
    shuffle=True, seed=seed, validate_filenames=False
)

val_flow = val_gen.flow_from_dataframe(
    df_val, x_col="filepath", y_col="label",
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode="binary", batch_size=batch_size,
    shuffle=False, validate_filenames=False
)

test_flow = test_gen.flow_from_dataframe(
    df_test_full, x_col="filepath", y_col=None,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode=None, batch_size=batch_size,
    shuffle=False, validate_filenames=False
)

# ==============================================================================
# 2. ARCHITECTURE TRANSFER LEARNING (MOBILENETV2)
# ==============================================================================
print("\n--- Initialisation de MobileNetV2 (Backbone gelée) ---")
base_model = MobileNetV2(input_shape=(IMG_HEIGHT, IMG_WIDTH, 3), include_top=False, weights='imagenet')
base_model.trainable = False  # On gèle les poids pré-entraînés

# Ajout de la nouvelle tête de classification adaptée à notre tâche binaire
x = GlobalAveragePooling2D()(base_model.output)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)  # Évite l'overfitting sur notre tête de réseau
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), # LR plus faible pour préserver les acquis
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# ==============================================================================
# 3. ENTRAÎNEMENT ET CALLBACKS
# ==============================================================================
early_stopping = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-6, verbose=1)

print("\n--- Début de l'entraînement ---")
history = model.fit(
    train_flow,
    epochs=15,
    validation_data=val_flow,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

# Courbes d'apprentissage
epochs_range = range(len(history.history['accuracy']))
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, history.history['accuracy'], label='Train Acc')
plt.plot(epochs_range, history.history['val_accuracy'], label='Val Acc')
plt.legend()
plt.title('Exactitude (Accuracy)')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history.history['loss'], label='Train Loss')
plt.plot(epochs_range, history.history['val_loss'], label='Val Loss')
plt.legend()
plt.title('Perte (Loss)')
plt.savefig("learning_curves.png")
print("[INFO] Graphique des courbes sauvegardé sous 'learning_curves.png'")

# ==============================================================================
# 4. ÉVALUATION ET MATRICE DE CONFUSION
# ==============================================================================
print("\n--- Évaluation du modèle sur la validation ---")
val_loss, val_acc = model.evaluate(val_flow)
print(f"Validation Loss: {val_loss:.4f} | Validation Accuracy: {val_acc:.4f}")

val_flow.reset()
val_preds = model.predict(val_flow)
val_pred_labels = (val_preds > 0.5).astype(int).flatten()
true_labels = val_flow.classes

print("\nMatrice de Confusion :")
print(confusion_matrix(true_labels, val_pred_labels))

print("\nRapport de classification complet :")
print(classification_report(true_labels, val_pred_labels, target_names=list(val_flow.class_indices.keys())))

# ==============================================================================
# 5. PRÉDICTIONS SUR LE JEU DE TEST (EXPORT CSV)
# ==============================================================================
print("\n--- Génération des prédictions sur le jeu de test non-étiqueté ---")
test_flow.reset()
test_preds = model.predict(test_flow)

df_submission = pd.DataFrame({
    "filepath": test_flow.filepaths,
    "prob_dog": test_preds.flatten(),
    "pred_label": ["dog" if p > 0.5 else "cat" for p in test_preds.flatten()]
})

df_submission.to_csv("submission_predictions.csv", index=False)
print("Prédictions exportées avec succès dans 'submission_predictions.csv'")

# ==============================================================================
# 6. SAUVEGARDE DES ARTIFACTS
# ==============================================================================
model.save("best_transfer_learning_model.h5")

metadata = {
    "architecture": "MobileNetV2 Transfer Learning",
    "img_dims": [IMG_HEIGHT, IMG_WIDTH],
    "batch_size": batch_size,
    "final_val_acc": float(val_acc),
    "class_indices": train_flow.class_indices
}

with open("training_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("Modèle enregistré sous 'best_transfer_learning_model.h5'")
print("Fichier de métadonnées généré sous 'training_metadata.json'")
print("\n🚀 Tout est prêt ! Prêt pour la soumission.")

ModuleNotFoundError: No module named 'tensorflow'